<a href="https://colab.research.google.com/github/bachaudhry/my-llm-from-scratch/blob/main/supplementary/pre-training/gpt-to-llama/converting_llama2_to_llama3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Converting Llama 2 to Llama 3

- In this notebook, we will code along with Sebastain's source material which converts Llama 2 to Llama 3 and Llama 3.1 in a step by step fashion.
- In depth details on both models can be found in the following papers:
 - [Llama 2: Open Foundation and Fine-Tuned Chat Models (2023)](https://arxiv.org/abs/2307.09288)
 - [The Llama 3 Herd of Models](https://arxiv.org/abs/2407.21783)

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/gpt-to-llama/gpt2-to-llama2-llama3.webp?1">

In [1]:
from importlib.metadata import version
pkgs = [
    "blobfile",         # to download pretrained weights
    "huggingface_hub",
    "matplotlib",
    "tiktoken",
    "torch",
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

blobfile version: 3.3.0
huggingface_hub version: 1.33.0
matplotlib version: 3.10.0
tiktoken version: 0.14.0
torch version: 2.11.0+cpu


# **1. Llama Model Conversion**

- The notebook [converting_gpt_to_llama2](https://github.com/bachaudhry/my-llm-from-scratch/blob/main/supplementary/pre-training/gpt-to-llama/converting_gpt_to_llama2.ipynb), recreated the implementation of Llama 2's components i.e. RMSNorm layers, SilU and SwiGLU activations, RoPE and SentencePiece tokenizer.
- Moving to the conversion of Llama 2 to Llama 3, we will specifically look at:
    - Modifications to the rotary embeddings
    - Implementing GQA.
    - Using a customized version of the GPT-4 tokenizer.

## **1.1 Reusing Llama 2 Components**

- Both Llama 2 and Llama 3 have lots of similarities so most of the building blocks can be imported from the notebook which dealt with the conversion of GPT 2 to Llama 2.

In [2]:
!git clone https://github.com/bachaudhry/my-llm-from-scratch.git

Cloning into 'my-llm-from-scratch'...
remote: Enumerating objects: 920, done.
remote: Counting objects: 100% (311/311), done.
remote: Compressing objects: 100% (275/275), done.
remote: Total 920 (delta 84), reused 70 (delta 24), pack-reused 609 (from 1)
Receiving objects: 100% (920/920), 2.09 MiB | 19.14 MiB/s, done.
Resolving deltas: 100% (465/465), done.


In [3]:
import os, sys, io, nbformat, types, ast

repo_folder = 'my-llm-from-scratch/supplementary/pre-training/gpt-to-llama'
os.chdir(repo_folder)
sys.path.append('.') # Add current directory of the repo to the path


In [21]:
import os, io, sys, types
import nbformat

import os, io, sys, types
from pathlib import Path
import nbformat


def _walk_files(root: Path, max_depth: int = 6):
    """Yield files under `root` up to `max_depth`, skipping build/hidden dirs."""
    skip = {".git", ".ipynb_checkpoints", "__pycache__", "node_modules", ".venv"}
    stack = [(root.resolve(), 0)]
    while stack:
        d, depth = stack.pop()
        if depth > max_depth:
            continue
        try:
            for p in d.iterdir():
                if p.name in skip:
                    continue
                if p.is_dir():
                    stack.append((p, depth + 1))
                else:
                    yield p
        except PermissionError:
            continue


def _find_notebook(fullname: str, repo_roots):
    """Locate `<fullname>.ipynb`, tolerating '-' vs '_' spelling differences."""
    # Both spellings, in priority order
    target_names = list(dict.fromkeys([
        fullname + ".ipynb",
        fullname.replace("-", "_") + ".ipynb",
        fullname.replace("_", "-") + ".ipynb",
    ]))

    roots = list(repo_roots or [])
    roots += [os.getcwd(), "/content"]
    roots += [p for p in sys.path if p and os.path.isdir(p)]
    roots = list(dict.fromkeys(roots))          # dedupe, keep order

    for root in roots:
        root_p = Path(root)
        if not root_p.exists():
            continue
        # 1. exact hit at root
        for name in target_names:
            direct = root_p / name
            if direct.is_file():
                return direct
        # 2. bounded recursive walk
        for p in _walk_files(root_p):
            if p.name in target_names:
                return p

    raise FileNotFoundError(
        f"'{fullname}.ipynb' not found. Roots searched: {roots}"
    )


def import_from_notebook(fullname, names, repo_roots=None, module_name=None):
    """
    Import selected functions / classes from an .ipynb notebook.

    Parameters
    ----------
    fullname : str
        Notebook basename without '.ipynb' (e.g. "converting_gpt_to_llama2").
    names : list[str]
        Function / class names to extract.
    repo_roots : list[str] | None
        Directories to search (e.g. the cloned repo root). Falls back to
        cwd, /content, and sys.path when not given.
    module_name : str | None
        Key under which the module is registered in sys.modules.
        Defaults to `fullname` with '-' -> '_'.

    Returns
    -------
    types.ModuleType
        A module whose namespace contains the requested definitions.
    """
    if module_name is None:
        module_name = fullname.replace("-", "_")

    nb_path = _find_notebook(fullname, repo_roots)
    print(f"[import_from_notebook] loading {nb_path}")

    with io.open(nb_path, "r", encoding="utf-8") as f:
        nb = nbformat.read(f, as_version=4)

    mod = types.ModuleType(module_name)
    mod.__file__ = str(nb_path)                 # helps with tracebacks / introspection
    sys.modules[module_name] = mod

    # Pre-seed names that notebook definitions almost always rely on, so we
    # don't depend on catching the notebook's own "import-only" cells.
    for prelude in (
        "import math",
        "import torch",
        "import torch.nn as nn",
        "import torch.nn.functional as F",
    ):
        exec(prelude, mod.__dict__)

    # Execute, in order, any cell that either defines a requested name or
    # contains an import statement. Each cell runs exactly once.
    for i, cell in enumerate(nb.cells):
        if cell.cell_type != "code":
            continue
        src = cell.source
        defines_target = any(
            (f"def {n}" in src) or (f"class {n}" in src) for n in names
        )
        is_import_cell = "import " in src
        if not (defines_target or is_import_cell):
            continue
        try:
            exec(compile(src, f"<{fullname}.ipynb cell {i}>", "exec"),
                 mod.__dict__)
        except Exception as e:
            print(f"[cell {i}] exec failed: {type(e).__name__}: {e}")

    missing = [n for n in names if not hasattr(mod, n)]
    if missing:
        print(f"[import_from_notebook] WARNING unresolved names: {missing}")

    return mod

In [22]:
imported_module = import_from_notebook(
    fullname="converting_gpt_to_llama2",
    names=["precompute_rope_params",
           "compute_rope",
           "SiLU",
           "FeedForward",
           "RMSNorm",
           "MultiHeadAttention"],
    repo_roots=["/content/my-llm-from-scratch"],
)

[import_from_notebook] loading /content/my-llm-from-scratch/supplementary/pre-training/gpt-to-llama/converting_gpt_to_llama2.ipynb
huggingface_hub version : 1.33.0
sentencepiece version : 0.2.2
torch version : 2.11.0+cpu


tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

[cell 47] exec failed: ModuleNotFoundError: No module named 'previous_chapters'
